In [3]:
import pandas as pd
import numpy as np
import os
import json
from search_info_by_gene import get_gene_info_by_gene_id

"""
场景假设：
1. 用户在前端输入框输入基因list
2. 前端传来一个json文件
3. 后端读取json文件，解析出基因list
4. 后端返回相关基因的表达谱
"""

def json_load(json_path):
    """读取json文件,转为字典
    """
    with open(json_path) as f:
        dict = json.load(f)

    return dict


def get_gene_list_form_json(json_path):
    """读取json文件,获取基因list
    """
    dict = json_load(json_path)
    gene_list = dict['gene_list']

    return gene_list


def get_gene_list_info(workdir,species_name,json_path):
    """根据基因list，获取相关基因的go注释
    """
    # 空格换成_，大写转小写
    species_name = species_name.replace(' ', '_').lower()

    # 获取基因list
    gene_list_info = []
    gene_list = get_gene_list_form_json(json_path)
    gene_list = [gene.upper() for gene in gene_list]

    for gene in gene_list:
        # 调用get_gene_info_by_gene_id函数，获取基因的go注释
        gene_info = get_gene_info_by_gene_id(workdir, species_name, gene)
        # 将每一个基因的go注释，添加到gene_list_info中
        gene_list_info.append(gene_info)

    return gene_list_info


def get_exp_df_json(workdir,species_name,json_path):
    """根据gene list，依据exp_gsm表，返回对应基因的表达谱
    """
    species_name = species_name.replace(' ', '_').lower()

    # 读取exp_gsm表
    exp_path = os.path.join(workdir, species_name, 'exp_gsm.tsv')
    df_exp = pd.read_csv(exp_path, sep='\t')

    # 获取基因list，转为大写
    gene_list = get_gene_list_form_json(json_path)
    gene_list = [gene.upper() for gene in gene_list]

    # 根据基因list，获取相关基因的表达量
    df = df_exp[df_exp['Gene id'].isin(gene_list)]
    df.iloc[:, 1:] = df.iloc[:, 1:].applymap(lambda x: np.log2(x + 1)).round(2)

    return df


# 根据基因list，根据Mt_Exp_fname的tsv表中获取相关基因的表达量
def get_exp_df_fname_json(workdir,species_name,json_path):
    """根据gene list，依据exp_fname表，返回对应基因的表达谱
    """
    species_name = species_name.replace(' ', '_').lower()

    # 读取exp_gsm表
    exp_path = os.path.join(workdir, species_name, 'exp_fname.tsv')
    df_exp = pd.read_csv(exp_path, sep='\t')

    # 获取基因list，转为大写
    gene_list = get_gene_list_form_json(json_path)
    gene_list = [gene.upper() for gene in gene_list]

    # 根据基因list，获取相关基因的表达量
    df = df_exp[df_exp['Gene id'].isin(gene_list)]
    df.iloc[:, 1:] = df.iloc[:, 1:].applymap(lambda x: np.log2(x + 1)).round(2)

    return df

In [7]:
# 工作路径
workdir = '~/mtd/transcriptome/search_exp'

# 假设前端传来的菌种名与通路名
species_name = 'Myceliophthora thermophila'
json_path = 'gene_list.json'

# get_gene_list_info(workdir, species_name, json_path)
# get_exp_df_json(workdir, species_name, json_path)

KeyError: 'Gene'